## Understanding Google's Gradient-weighted Class Activation Mapping

Grad-CAM is a visual explanation technique for deep convolution networks (CNNs) that highlights image regions most relevant to a specific prediction. In essence, given a target concept or class (e.g. “cat”), Grad-CAM traces back the gradient of the class score to the final convolutional layer. By pooling those gradients, it assigns an “importance weight” to each feature map (convolutional channel) and combines them to form a coarse heatmap of salient regions. In practice this produces a class-discriminative localization map: hot regions in the image indicate parts that most influence the model’s decision for the chosen class. 

## How Grad-CAM Works 
The intuition behind Grad-CAM is that the gradient of the class score $y^c$ with respect to a feature map $A^k$ tells us how much a small change in $A^k$ would affect the score for class $c$. Gradients flowing back into the last convolutional layer indicate which neurons and spatial locations most influence the decision.
### Steps 
### 1. **Forward Pass**: <br>
Input the image into the CNN and compute the score (logit) $y^c^ for the target class $c$. Record the feature maps $A^k$ from the last convolutional layer.
### 2. **Backward Pass**: <br>
Compute the partial derivaties/gradients $\frac{\partial y^c}{\partial A^k_{ij}}$ for each spatial location $(i,j)$ and feature channel $k$. In practice this is done by backpropagating the gradient of the score for class $c$ while zeroing out all other classes​.
### 3. **Channel Weighting**: <br>
Compute a weight $\alpha_k^c$ for each feature map by global-average-pooling the gradients over the spatial dimensions:
$$\alpha_k^c = \frac{1}{Z} \sum_{i,j} \frac{\partial y^c}{\partial A^k_{ij}}$$
where $Z$ is the number of pixels ($Z = u\times v$ for a feature map of size $u\times v$)​. Intuitively, $\alpha_k^c$ is the importance of feature map $k$ for class $c$: if increasing $A^k_{ij}$ tends to increase $y^c$, then $\alpha_k^c$ will be large.<br>
### 4. **Weighted Combination**: <br>
Form a raw class activation map by a linear combination of the forward feature maps, using the weights $\alpha_k^c$:
$$L^c_{Grad-CAM} = ReLU\left(\sum_k \alpha_k^c A^k\right)$$
whereThe sum $\sum_k \alpha_k^c A^k$ is computed elementwise over each pixel; then a ReLU is applied to keep only positive influences (since negative values would decrease the class score)​. This ensures the heatmap highlights only features that positively support class $c$.
### 5. **Heatmap**<br>
 The resulting map $L^c_{\text{Grad-CAM}}$ is a coarse heatmap at the same spatial resolution as the feature maps (e.g. $14\times14$ for VGG/AlexNet conv layers)​. Finally, upsample $L^c_{\text{Grad-CAM}}$ (e.g. via bilinear interpolation) to the original image size​ and overlay it on the image. Areas with higher values (often shown in red) indicate regions most important for the network’s prediction of class $c$​

## Code Demonstration